# Practice training a deep neural network on the CIFAR10 image dataset:

## a. 
Load CIFAR10 just like you loaded the FashionMNIST dataset in Chapter 10, but using torchvision.datasets.CIFAR10 instead of. The
dataset is composed of 60,000 32 × 32–pixel color images (50,000 for training, 10,000 for testing) with 10 classes.

In [1]:
import sys
assert sys.version_info >= (3, 10)

In [2]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [4]:
import torch
import torch.nn as nn
import torchmetrics
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import DataLoader


### Import del dataset y división entre train, validate y test (45k - 5k)

In [5]:
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.CIFAR10(
root="datasets", train=True, download=True, transform=toTensor)

test_data = torchvision.datasets.CIFAR10(
root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(42)
learning_rate = 0.002 

train_data, valid_data = torch.utils.data.random_split(
train_and_valid_data, [45_000, 5_000])

/home/danielbb/Documentos/Proyectos/ML-Roadmap/ApuntesML/Practica/NN_DL/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


### Creación de los dataloaders para el entrenamiento

In [6]:
batch_size = 128
train_loader = DataLoader(train_data, batch_size=batch_size)
eval_loader = DataLoader(valid_data, batch_size=batch_size)
test_loader = DataLoader(test_data, batch_size=batch_size)

## b.
Build a DNN with 20 hidden layers of 100 neurons each (that’s too many, but it’s the point of this exercise). Use He initialization and the Swish activation function (using nn.SiLU). Since this is a classification task, you will need an output layer with one neuron per class

### Definición de la función para inicialización HE

In [7]:
def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        #nn.init.zeros_(module.bias)

### Creación del modelo

In [8]:

entrada = nn.Linear(3 * 32 * 32, 100)
salida = nn.Linear(100, 10)
capas = []
for i in range(19):
    capas.append(nn.Linear(100, 100))
    capas.append(nn.SiLU())
capas.insert(0, nn.SiLU())
capas.insert(0,entrada)
capas.insert(0,nn.Flatten())
capas.append(salida)
modelo = nn.Sequential(*capas)
modelo.apply(use_he_init)
modelo.to(device)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=3072, out_features=100, bias=True)
  (2): SiLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): SiLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): SiLU()
  (7): Linear(in_features=100, out_features=100, bias=True)
  (8): SiLU()
  (9): Linear(in_features=100, out_features=100, bias=True)
  (10): SiLU()
  (11): Linear(in_features=100, out_features=100, bias=True)
  (12): SiLU()
  (13): Linear(in_features=100, out_features=100, bias=True)
  (14): SiLU()
  (15): Linear(in_features=100, out_features=100, bias=True)
  (16): SiLU()
  (17): Linear(in_features=100, out_features=100, bias=True)
  (18): SiLU()
  (19): Linear(in_features=100, out_features=100, bias=True)
  (20): SiLU()
  (21): Linear(in_features=100, out_features=100, bias=True)
  (22): SiLU()
  (23): Linear(in_features=100, out_features=100, bias=True)
  (24): SiLU()
  (25): Linear(in_features=100, out_features

## c.
Using NAdam optimization and early stopping, train the network on the CIFAR10 dataset. Remember to search for the right learning rate each time you change the model’s architecture or hyperparameters.

In [32]:
import copy
def entrenar(modelo, optimizador, criterio, accuracy, train_loader, eval_loader, n_iter, device, min_delta = 0.001, paciencia=6, scheduler = None):
    # Early stopping:
    # min_delta: mejora mínima exigida en val_loss para considerar que realmente mejoró
    # patience: número de épocas consecutivas sin mejora significativa antes de parar
    best_val_acc = 0.0
    patience_counter = 0
    
    # Guardamos el mejor estado del modelo para restaurarlo al final
    best_state = copy.deepcopy(modelo.state_dict())
    
    eval_calc = None


    for i in range(n_iter):
        modelo.train()
        
        total_loss = 0.0
        total_train_samples = 0
        for sample, target in train_loader:
            sample_batch = sample.to(device)
            target_batch = target.to(device)
            # Optimizador a 0 para evitar la acumulación
            optimizador.zero_grad()

            pred = modelo(sample_batch)
            perdida = criterio(pred, target_batch)
            perdida.backward()
            optimizador.step()
            
            
            # Acumulamos pérdida ponderada por tamaño de batch
            bs = target_batch.size(0)
            total_loss += perdida.item() * bs
            total_train_samples += bs
    
        mean_loss = total_loss / total_train_samples
        print(f"Epoch {i + 1}/{n_iter}, Loss: {mean_loss:.4f}")
            

        accuracy.reset()
        
        modelo.eval()

        val_loss_sum = 0.0
        total_val_samples = 0
        
        with torch.no_grad():
            for sample, target in eval_loader:
                sample_batch = sample.to(device)
                target_batch = target.to(device)
                pred = modelo(sample_batch)
                perdida = criterio(pred, target_batch)
                accuracy.update(pred, target_batch)

                # Acumulamos pérdida ponderada por tamaño de batch para cálculo más correcto
                bs = target_batch.size(0)
                val_loss_sum += perdida.item() * bs
                total_val_samples += bs

        mean_val_loss = val_loss_sum / total_val_samples
        val_acc = accuracy.compute()
        print(f"Epoch {i + 1}/{n_iter}, Val Loss: {mean_val_loss:.4f}, Val Acc: {val_acc:.4f}")

        if scheduler is not None:
                scheduler.step()
            
        # Early stopping sobre val_acc:
        # Solo consideramos mejora si sube al menos min_delta respecto al mejor valor histórico
        if val_acc > (best_val_acc + min_delta):
            best_val_acc = val_acc
            best_state = copy.deepcopy(modelo.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"EarlyStopping: {patience_counter}/{paciencia} sin mejora significativa")
            
                
            if patience_counter >= paciencia:
                print(f"Parada por early stopping. Mejor val_loss: {best_val_acc:.4f}")
                break
    modelo.load_state_dict(best_state) 
    return modelo, best_val_acc
            

In [10]:
import time

def train_libro(model, optimizer, loss_fn, metric, train_loader,
                              valid_loader, n_epochs, patience=10,
                              checkpoint_path=None, scheduler=None):
    checkpoint_path = checkpoint_path or "my_checkpoint.pt"
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    best_metric = 0.0
    patience_counter = 0
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        t0 = time.time()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)

        train_metric = metric.compute().item()
        valid_metric = evaluate_tm(model, valid_loader, metric).item()
        if valid_metric > best_metric:
            torch.save(model.state_dict(), checkpoint_path)
            best_metric = valid_metric
            best = " (best)"
            patience_counter = 0
        else:
            patience_counter += 1
            best = ""

        t1 = time.time()
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(valid_metric)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}{best}"
              f" in {t1 - t0:.1f}s"
        )
        if scheduler is not None:
            scheduler.step()
        if patience_counter >= patience:
            print("Early stopping!")
            break

    model.load_state_dict(torch.load(checkpoint_path))
    return history

In [11]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [12]:
optimizer = torch.optim.NAdam(modelo.parameters(), betas=(0.9, 0.999), lr=0.002)
exp_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
entrenar(modelo, optimizer, xentropy,  accuracy, train_loader, eval_loader, 50, device, scheduler=exp_scheduler)
#train_libro(modelo, optimizer, xentropy, accuracy, train_loader, eval_loader, 50, scheduler=cosine_repeat_scheduler)

Epoch 1/50, Loss: 2.2788
Epoch 1/50, Val Loss: 2.1325, Val Acc: 0.1652
Epoch 2/50, Loss: 2.0814
Epoch 2/50, Val Loss: 2.0504, Val Acc: 0.2302
Epoch 3/50, Loss: 1.9730
Epoch 3/50, Val Loss: 1.9214, Val Acc: 0.2760
Epoch 4/50, Loss: 1.9122
Epoch 4/50, Val Loss: 1.9282, Val Acc: 0.2684
EarlyStopping: 1/6 sin mejora significativa
Epoch 5/50, Loss: 1.8887
Epoch 5/50, Val Loss: 1.8616, Val Acc: 0.3018
Epoch 6/50, Loss: 1.8440
Epoch 6/50, Val Loss: 1.8384, Val Acc: 0.3098
Epoch 7/50, Loss: 1.8197
Epoch 7/50, Val Loss: 1.8500, Val Acc: 0.3088
EarlyStopping: 1/6 sin mejora significativa
Epoch 8/50, Loss: 1.8035
Epoch 8/50, Val Loss: 1.8065, Val Acc: 0.3238
Epoch 9/50, Loss: 1.7763
Epoch 9/50, Val Loss: 1.7949, Val Acc: 0.3298
Epoch 10/50, Loss: 1.7536
Epoch 10/50, Val Loss: 1.7646, Val Acc: 0.3556
Epoch 11/50, Loss: 1.7181
Epoch 11/50, Val Loss: 1.7550, Val Acc: 0.3638
Epoch 12/50, Loss: 1.6880
Epoch 12/50, Val Loss: 1.7238, Val Acc: 0.3752
Epoch 13/50, Loss: 1.6625
Epoch 13/50, Val Loss: 1.707

(Sequential(
   (0): Flatten(start_dim=1, end_dim=-1)
   (1): Linear(in_features=3072, out_features=100, bias=True)
   (2): SiLU()
   (3): Linear(in_features=100, out_features=100, bias=True)
   (4): SiLU()
   (5): Linear(in_features=100, out_features=100, bias=True)
   (6): SiLU()
   (7): Linear(in_features=100, out_features=100, bias=True)
   (8): SiLU()
   (9): Linear(in_features=100, out_features=100, bias=True)
   (10): SiLU()
   (11): Linear(in_features=100, out_features=100, bias=True)
   (12): SiLU()
   (13): Linear(in_features=100, out_features=100, bias=True)
   (14): SiLU()
   (15): Linear(in_features=100, out_features=100, bias=True)
   (16): SiLU()
   (17): Linear(in_features=100, out_features=100, bias=True)
   (18): SiLU()
   (19): Linear(in_features=100, out_features=100, bias=True)
   (20): SiLU()
   (21): Linear(in_features=100, out_features=100, bias=True)
   (22): SiLU()
   (23): Linear(in_features=100, out_features=100, bias=True)
   (24): SiLU()
   (25): Linear(in

## d
Now try adding batch-norm and compare the learning curves: is it converging faster than before? Does it produce a better model? How does it affect training speed?

In [13]:
torch.manual_seed(69)

capas = [nn.Flatten(), 
         nn.Linear(3 * 32 * 32, 100), 
         nn.BatchNorm1d(100), 
         nn.SiLU()]

for i in range(19):
    capas.append(nn.Linear(100, 100))
    capas.append(nn.BatchNorm1d(100))
    capas.append(nn.SiLU())

capas.append(nn.Linear(100, 10))

modelo_bn = nn.Sequential(*capas)
modelo_bn.apply(use_he_init)
modelo_bn.to(device)

optimizer2 = torch.optim.NAdam(modelo_bn.parameters(), betas=(0.9, 0.999), lr=0.002)
exp_scheduler2 = torch.optim.lr_scheduler.ExponentialLR(optimizer2, gamma=0.9)


In [14]:
entrenar(modelo_bn, optimizer2, xentropy,  accuracy, train_loader, eval_loader, 50, device, scheduler=exp_scheduler2)


Epoch 1/50, Loss: 1.9400
Epoch 1/50, Val Loss: 1.7648, Val Acc: 0.3564
Epoch 2/50, Loss: 1.6265
Epoch 2/50, Val Loss: 1.7554, Val Acc: 0.3824
Epoch 3/50, Loss: 1.4982
Epoch 3/50, Val Loss: 1.7491, Val Acc: 0.3806
Epoch 4/50, Loss: 1.4023
Epoch 4/50, Val Loss: 1.7190, Val Acc: 0.3848
Epoch 5/50, Loss: 1.3234
Epoch 5/50, Val Loss: 1.7757, Val Acc: 0.3878
EarlyStopping: 1/6 sin mejora significativa
Epoch 6/50, Loss: 1.2553
Epoch 6/50, Val Loss: 1.7381, Val Acc: 0.3976
EarlyStopping: 2/6 sin mejora significativa
Epoch 7/50, Loss: 1.1928
Epoch 7/50, Val Loss: 1.7082, Val Acc: 0.4210
Epoch 8/50, Loss: 1.1322
Epoch 8/50, Val Loss: 1.7382, Val Acc: 0.4190
EarlyStopping: 1/6 sin mejora significativa
Epoch 9/50, Loss: 1.0752
Epoch 9/50, Val Loss: 1.8751, Val Acc: 0.4036
EarlyStopping: 2/6 sin mejora significativa
Epoch 10/50, Loss: 1.0221
Epoch 10/50, Val Loss: 1.9090, Val Acc: 0.4112
EarlyStopping: 3/6 sin mejora significativa
Epoch 11/50, Loss: 0.9662
Epoch 11/50, Val Loss: 2.0040, Val Acc: 0.

(Sequential(
   (0): Flatten(start_dim=1, end_dim=-1)
   (1): Linear(in_features=3072, out_features=100, bias=True)
   (2): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (3): SiLU()
   (4): Linear(in_features=100, out_features=100, bias=True)
   (5): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (6): SiLU()
   (7): Linear(in_features=100, out_features=100, bias=True)
   (8): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (9): SiLU()
   (10): Linear(in_features=100, out_features=100, bias=True)
   (11): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (12): SiLU()
   (13): Linear(in_features=100, out_features=100, bias=True)
   (14): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (15): SiLU()
   (16): Linear(in_features=100, out_features=100, bias=True)
   (17): BatchNorm1d(100, eps=1e-05, momentum=0.1, a

## e. 
Try replacing batch-norm with SELU, and make the necessary adjustments to ensure the network self-normalizes (i.e., standardize the input features, use LeCun normal initialization, make sure the DNN contains only a sequence of dense layers, etc.).

In [15]:
def use_lecun_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, mode="fan_in",
                                nonlinearity="linear")
        nn.init.zeros_(module.bias)

In [16]:
torch.manual_seed(69)

capas = [nn.Flatten(), 
         nn.Linear(3 * 32 * 32, 100), 
         nn.BatchNorm1d(100), 
         nn.SiLU()]

for i in range(19):
    capas.append(nn.Linear(100, 100))
    capas.append(nn.BatchNorm1d(100))
    capas.append(nn.SELU())

capas.append(nn.Linear(100, 10))

modelo_selu = nn.Sequential(*capas)
modelo_selu.apply(use_lecun_init)
modelo_selu.to(device)

optimizer3 = torch.optim.NAdam(modelo_selu.parameters(), betas=(0.9, 0.999), lr=0.002)
exp_scheduler3 = torch.optim.lr_scheduler.ExponentialLR(optimizer3, gamma=0.9)


In [17]:
from torch.utils.data import DataLoader
def standarize_img(loader):   
    mean = torch.zeros(3)
    std = torch.zeros(3)
    total_samples = 0
    
    for images, _ in loader:
        batch_samples = images.size(0)
        images = images.view(batch_samples, images.size(1), -1)  # (B, C, H*W)
        mean += images.mean(dim=2).sum(dim=0)
        std += images.std(dim=2).sum(dim=0)
        total_samples += batch_samples
    
    mean /= total_samples
    std /= total_samples

    return mean, std

mean, std = standarize_img(train_loader)

toTensor_norm = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=mean.tolist(), std=std.tolist())
])

train_and_valid_data.transform = toTensor_norm
test_data.transform = toTensor_norm

batch_size = 128
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(valid_data, batch_size=batch_size)
test_loader = DataLoader(test_data, batch_size=batch_size)


In [18]:
entrenar(modelo_selu, optimizer3, xentropy,  accuracy, train_loader, eval_loader, 50, device, scheduler=exp_scheduler3)


Epoch 1/50, Loss: 1.7740
Epoch 1/50, Val Loss: 1.7469, Val Acc: 0.3866
Epoch 2/50, Loss: 1.5680
Epoch 2/50, Val Loss: 1.5772, Val Acc: 0.4406
Epoch 3/50, Loss: 1.4682
Epoch 3/50, Val Loss: 1.5323, Val Acc: 0.4620
Epoch 4/50, Loss: 1.3944
Epoch 4/50, Val Loss: 1.5663, Val Acc: 0.4462
EarlyStopping: 1/6 sin mejora significativa
Epoch 5/50, Loss: 1.3299
Epoch 5/50, Val Loss: 1.4645, Val Acc: 0.4866
Epoch 6/50, Loss: 1.2778
Epoch 6/50, Val Loss: 1.4965, Val Acc: 0.4802
EarlyStopping: 1/6 sin mejora significativa
Epoch 7/50, Loss: 1.2340
Epoch 7/50, Val Loss: 1.3973, Val Acc: 0.5106
Epoch 8/50, Loss: 1.1907
Epoch 8/50, Val Loss: 1.3944, Val Acc: 0.5044
Epoch 9/50, Loss: 1.1501
Epoch 9/50, Val Loss: 1.4042, Val Acc: 0.5118
EarlyStopping: 1/6 sin mejora significativa
Epoch 10/50, Loss: 1.1129
Epoch 10/50, Val Loss: 1.3781, Val Acc: 0.5226
Epoch 11/50, Loss: 1.0804
Epoch 11/50, Val Loss: 1.3573, Val Acc: 0.5296
Epoch 12/50, Loss: 1.0409
Epoch 12/50, Val Loss: 1.3914, Val Acc: 0.5152
EarlyStopp

(Sequential(
   (0): Flatten(start_dim=1, end_dim=-1)
   (1): Linear(in_features=3072, out_features=100, bias=True)
   (2): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (3): SiLU()
   (4): Linear(in_features=100, out_features=100, bias=True)
   (5): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (6): SELU()
   (7): Linear(in_features=100, out_features=100, bias=True)
   (8): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (9): SELU()
   (10): Linear(in_features=100, out_features=100, bias=True)
   (11): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (12): SELU()
   (13): Linear(in_features=100, out_features=100, bias=True)
   (14): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (15): SELU()
   (16): Linear(in_features=100, out_features=100, bias=True)
   (17): BatchNorm1d(100, eps=1e-05, momentum=0.1, a

## f

Try regularizing the model with alpha dropout. Then, without retraining your model, see if you can achieve better accuracy using MC dropout.

In [50]:
torch.manual_seed(69)

capas = [nn.Flatten(), 
         nn.Linear(3 * 32 * 32, 100), 
         nn.SELU(),
         nn.AlphaDropout(p=0.1)]

for i in range(19):
    capas.append(nn.Linear(100, 100))
    capas.append(nn.SELU())
    capas.append(nn.AlphaDropout(p=0.1))

capas.append(nn.Linear(100, 10))

modelo_alpha_do = nn.Sequential(*capas)
modelo_alpha_do.apply(use_lecun_init)
modelo_alpha_do.to(device)

optimizer4 = torch.optim.NAdam(modelo_alpha_do.parameters(), lr=0.002)
exp_scheduler4 = torch.optim.lr_scheduler.ExponentialLR(optimizer4, gamma=0.9)
xentropy = nn.CrossEntropyLoss(label_smoothing=0.1)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)


In [51]:
torch.manual_seed(0)
entrenar(modelo_alpha_do, optimizer4, xentropy,  accuracy, train_loader, eval_loader, 50, device, scheduler = exp_scheduler4)


Epoch 1/50, Loss: 2.1846
Epoch 1/50, Val Loss: 3.2026, Val Acc: 0.1848
Epoch 2/50, Loss: 2.0392
Epoch 2/50, Val Loss: 3.9260, Val Acc: 0.2534
Epoch 3/50, Loss: 1.9900
Epoch 3/50, Val Loss: 5.0668, Val Acc: 0.2712
Epoch 4/50, Loss: 1.9575
Epoch 4/50, Val Loss: 4.4227, Val Acc: 0.2984
Epoch 5/50, Loss: 1.9432
Epoch 5/50, Val Loss: 4.4064, Val Acc: 0.3026
Epoch 6/50, Loss: 1.9129
Epoch 6/50, Val Loss: 4.5827, Val Acc: 0.3004
EarlyStopping: 1/6 sin mejora significativa
Epoch 7/50, Loss: 1.8937
Epoch 7/50, Val Loss: 4.3063, Val Acc: 0.3266
Epoch 8/50, Loss: 1.8900
Epoch 8/50, Val Loss: 4.7378, Val Acc: 0.3318
Epoch 9/50, Loss: 1.8690
Epoch 9/50, Val Loss: 4.6060, Val Acc: 0.3386
Epoch 10/50, Loss: 1.8384
Epoch 10/50, Val Loss: 4.4703, Val Acc: 0.3580
Epoch 11/50, Loss: 1.8238
Epoch 11/50, Val Loss: 4.8168, Val Acc: 0.3362
EarlyStopping: 1/6 sin mejora significativa
Epoch 12/50, Loss: 1.8011
Epoch 12/50, Val Loss: 4.7508, Val Acc: 0.3580
EarlyStopping: 2/6 sin mejora significativa
Epoch 13/5

(Sequential(
   (0): Flatten(start_dim=1, end_dim=-1)
   (1): Linear(in_features=3072, out_features=100, bias=True)
   (2): SELU()
   (3): AlphaDropout(p=0.1, inplace=False)
   (4): Linear(in_features=100, out_features=100, bias=True)
   (5): SELU()
   (6): AlphaDropout(p=0.1, inplace=False)
   (7): Linear(in_features=100, out_features=100, bias=True)
   (8): SELU()
   (9): AlphaDropout(p=0.1, inplace=False)
   (10): Linear(in_features=100, out_features=100, bias=True)
   (11): SELU()
   (12): AlphaDropout(p=0.1, inplace=False)
   (13): Linear(in_features=100, out_features=100, bias=True)
   (14): SELU()
   (15): AlphaDropout(p=0.1, inplace=False)
   (16): Linear(in_features=100, out_features=100, bias=True)
   (17): SELU()
   (18): AlphaDropout(p=0.1, inplace=False)
   (19): Linear(in_features=100, out_features=100, bias=True)
   (20): SELU()
   (21): AlphaDropout(p=0.1, inplace=False)
   (22): Linear(in_features=100, out_features=100, bias=True)
   (23): SELU()
   (24): AlphaDropout(

In [52]:
# Dropout Monte Carlo
# Definimos el modo evaluacion antes de hacer las predicciones
modelo_alpha_do.eval()
# Pero iteramos todas las capas de la red neuronal para poner en modo entrenamiento aquellas que sean dropout
for module in modelo_alpha_do.modules():
    if isinstance(module, nn.Dropout):
        module.train()



temp_loader = DataLoader(test_data, batch_size=10, shuffle=False)
X_new, y_new = next(iter(temp_loader))
X_new = X_new.to(device)
torch.manual_seed(42)

with torch.no_grad():
    # Generamos un lote de 100 imagenes de la misma imagen con repeat_interleave(), con forma [300, 1, 28, 28] 
    # ya que estan todas en la primera dimension
    X_new_repeated = X_new.repeat_interleave(100, dim=0)
    # El modelo genera predicciones con 10 logits por imagen con una forma de tensor [300, 10],
    # el cual reestructuramos a [3, 100, 10] para agrupar las predicciones para cada imagen
    y_logits_all = modelo_alpha_do(X_new_repeated).reshape(10, 100, 10)
    # Transformamos los logits en probabilidades con la funcion softmax
    y_probas_all = torch.nn.functional.softmax(y_logits_all, dim=-1)
    # Calculamos la media de la segunda dimension para obtener la pobabilidad media estimada
    # de cada una de las clases en cada una de las imagenes con resultado de un tensor [3, 10]
    y_probas = y_probas_all.mean(dim=1)
    # Calculamos tambien la desviacion estandar entre las 100 muestras, como medida de incertidumbre
    y_std = y_probas_all.std(dim=1)  # forma [3, 10]
    # Obtenemos la clase predicha (mayor probabilidad media) para cada imagen
    y_pred = y_probas.argmax(dim=1)  # forma [3]

# Comprobamos los resultados fuera del no_grad, ya que solo es lectura de tensores ya calculados
for i in range(len(y_pred)):
    clase_pred = y_pred[i].item()
    clase_real = y_new[i].item()
    prob_media = y_probas[i, clase_pred].item()
    incertidumbre = y_std[i, clase_pred].item()
    acierto = "✓" if clase_pred == clase_real else "✗"
    print(f"Imagen {i}: predicha={clase_pred}, real={clase_real} {acierto} | "
          f"prob. media={prob_media:.3f}, std={incertidumbre:.3f}")


Imagen 0: predicha=5, real=3 ✗ | prob. media=0.964, std=0.000
Imagen 1: predicha=1, real=8 ✗ | prob. media=0.933, std=0.000
Imagen 2: predicha=8, real=8 ✓ | prob. media=0.985, std=0.000
Imagen 3: predicha=8, real=0 ✗ | prob. media=0.573, std=0.000
Imagen 4: predicha=6, real=6 ✓ | prob. media=0.804, std=0.000
Imagen 5: predicha=6, real=6 ✓ | prob. media=0.998, std=0.000
Imagen 6: predicha=5, real=1 ✗ | prob. media=1.000, std=0.000
Imagen 7: predicha=6, real=6 ✓ | prob. media=0.831, std=0.000
Imagen 8: predicha=5, real=3 ✗ | prob. media=0.716, std=0.000
Imagen 9: predicha=1, real=1 ✓ | prob. media=0.999, std=0.000


## g 

Retrain your model using 1cycle scheduling and see if it improves training speed and model accuracy.

In [54]:
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer4, epochs=50, steps_per_epoch=len(train_loader), max_lr=1e-2)
entrenar(modelo_alpha_do, optimizer4, xentropy,  accuracy, train_loader, eval_loader, 50, device, scheduler = scheduler)


Epoch 1/50, Loss: 1.6634
Epoch 1/50, Val Loss: 3.7762, Val Acc: 0.4102
Epoch 2/50, Loss: 1.6666
Epoch 2/50, Val Loss: 4.1527, Val Acc: 0.4172
Epoch 3/50, Loss: 1.6666
Epoch 3/50, Val Loss: 3.5911, Val Acc: 0.4106
EarlyStopping: 1/6 sin mejora significativa
Epoch 4/50, Loss: 1.6531
Epoch 4/50, Val Loss: 3.3955, Val Acc: 0.4172
EarlyStopping: 2/6 sin mejora significativa
Epoch 5/50, Loss: 1.6486
Epoch 5/50, Val Loss: 3.9119, Val Acc: 0.4214
Epoch 6/50, Loss: 1.6449
Epoch 6/50, Val Loss: 3.8035, Val Acc: 0.4190
EarlyStopping: 1/6 sin mejora significativa
Epoch 7/50, Loss: 1.6459
Epoch 7/50, Val Loss: 3.6817, Val Acc: 0.4176
EarlyStopping: 2/6 sin mejora significativa
Epoch 8/50, Loss: 1.6361
Epoch 8/50, Val Loss: 4.0582, Val Acc: 0.4168
EarlyStopping: 3/6 sin mejora significativa
Epoch 9/50, Loss: 1.6269
Epoch 9/50, Val Loss: 3.6040, Val Acc: 0.4184
EarlyStopping: 4/6 sin mejora significativa
Epoch 10/50, Loss: 1.6245
Epoch 10/50, Val Loss: 3.7602, Val Acc: 0.4144
EarlyStopping: 5/6 sin m

(Sequential(
   (0): Flatten(start_dim=1, end_dim=-1)
   (1): Linear(in_features=3072, out_features=100, bias=True)
   (2): SELU()
   (3): AlphaDropout(p=0.1, inplace=False)
   (4): Linear(in_features=100, out_features=100, bias=True)
   (5): SELU()
   (6): AlphaDropout(p=0.1, inplace=False)
   (7): Linear(in_features=100, out_features=100, bias=True)
   (8): SELU()
   (9): AlphaDropout(p=0.1, inplace=False)
   (10): Linear(in_features=100, out_features=100, bias=True)
   (11): SELU()
   (12): AlphaDropout(p=0.1, inplace=False)
   (13): Linear(in_features=100, out_features=100, bias=True)
   (14): SELU()
   (15): AlphaDropout(p=0.1, inplace=False)
   (16): Linear(in_features=100, out_features=100, bias=True)
   (17): SELU()
   (18): AlphaDropout(p=0.1, inplace=False)
   (19): Linear(in_features=100, out_features=100, bias=True)
   (20): SELU()
   (21): AlphaDropout(p=0.1, inplace=False)
   (22): Linear(in_features=100, out_features=100, bias=True)
   (23): SELU()
   (24): AlphaDropout(

In [55]:
# Dropout Monte Carlo
# Definimos el modo evaluacion antes de hacer las predicciones
modelo_alpha_do.eval()
# Pero iteramos todas las capas de la red neuronal para poner en modo entrenamiento aquellas que sean dropout
for module in modelo_alpha_do.modules():
    if isinstance(module, nn.Dropout):
        module.train()



temp_loader = DataLoader(test_data, batch_size=10, shuffle=False)
X_new, y_new = next(iter(temp_loader))
X_new = X_new.to(device)
torch.manual_seed(42)

with torch.no_grad():
    # Generamos un lote de 100 imagenes de la misma imagen con repeat_interleave(), con forma [300, 1, 28, 28] 
    # ya que estan todas en la primera dimension
    X_new_repeated = X_new.repeat_interleave(100, dim=0)
    # El modelo genera predicciones con 10 logits por imagen con una forma de tensor [300, 10],
    # el cual reestructuramos a [3, 100, 10] para agrupar las predicciones para cada imagen
    y_logits_all = modelo_alpha_do(X_new_repeated).reshape(10, 100, 10)
    # Transformamos los logits en probabilidades con la funcion softmax
    y_probas_all = torch.nn.functional.softmax(y_logits_all, dim=-1)
    # Calculamos la media de la segunda dimension para obtener la pobabilidad media estimada
    # de cada una de las clases en cada una de las imagenes con resultado de un tensor [3, 10]
    y_probas = y_probas_all.mean(dim=1)
    # Calculamos tambien la desviacion estandar entre las 100 muestras, como medida de incertidumbre
    y_std = y_probas_all.std(dim=1)  # forma [3, 10]
    # Obtenemos la clase predicha (mayor probabilidad media) para cada imagen
    y_pred = y_probas.argmax(dim=1)  # forma [3]

# Comprobamos los resultados fuera del no_grad, ya que solo es lectura de tensores ya calculados
for i in range(len(y_pred)):
    clase_pred = y_pred[i].item()
    clase_real = y_new[i].item()
    prob_media = y_probas[i, clase_pred].item()
    incertidumbre = y_std[i, clase_pred].item()
    acierto = "✓" if clase_pred == clase_real else "✗"
    print(f"Imagen {i}: predicha={clase_pred}, real={clase_real} {acierto} | "
          f"prob. media={prob_media:.3f}, std={incertidumbre:.3f}")



Imagen 0: predicha=5, real=3 ✗ | prob. media=0.581, std=0.000
Imagen 1: predicha=1, real=8 ✗ | prob. media=0.961, std=0.000
Imagen 2: predicha=8, real=8 ✓ | prob. media=0.988, std=0.000
Imagen 3: predicha=8, real=0 ✗ | prob. media=0.601, std=0.000
Imagen 4: predicha=6, real=6 ✓ | prob. media=0.871, std=0.000
Imagen 5: predicha=6, real=6 ✓ | prob. media=1.000, std=0.000
Imagen 6: predicha=5, real=1 ✗ | prob. media=0.986, std=0.000
Imagen 7: predicha=6, real=6 ✓ | prob. media=1.000, std=0.000
Imagen 8: predicha=5, real=3 ✗ | prob. media=0.591, std=0.000
Imagen 9: predicha=1, real=1 ✓ | prob. media=0.858, std=0.000
